El presente notebook muestra los resultados de la implementación de un modelo de redes neruonales convolucionales simple, con un total de 6 capas.

Importación de librerías

In [2]:
import os
import numpy as np
from PIL import Image
import tensorflow as tf
from sklearn.model_selection import train_test_split  # Importa función para dividir datos en entrenamiento y prueba
from tensorflow.keras.models import Model, Sequential # Importa la clase Model de Keras
from tensorflow.keras.layers import Conv2D, Input, BatchNormalization,MaxPooling2D, Flatten, Dense, Dropout, UpSampling2D  # Importa capas de Keras para construir el modelo
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint  # Importa el callback
from tensorflow.keras.metrics import RootMeanSquaredError, MeanAbsolutePercentageError
import matplotlib.pyplot as plt

2025-10-06 10:25:16.004310: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2025-10-06 10:25:16.005760: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-10-06 10:25:16.061505: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-10-06 10:25:17.132959: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off,

Funciones

In [3]:
# Directorio original y reducido
d_original = '/media/smoke/SSD_D0/bandas/banda13/ene_01_mes'
d_reducida = '/media/smoke/SSD_D0/bandas/banda13/ene_01_reducida_mes'


# Función de redimensionado
def resize_npy_images(d_original, d_reducida, new_size):  # Define función para redimensionar imágenes .npy
    if not os.path.exists(d_reducida):  # Si el directorio reducido no existe
        os.makedirs(d_reducida)  # Lo crea
    for filename in os.listdir(d_original):  # Recorre los archivos en el directorio original
        if filename.lower().endswith('.npy'):  # Si el archivo es .npy
            img_path = os.path.join(d_original, filename)  # Construye la ruta completa del archivo
            img_array = np.load(img_path)  # Carga el array de la imagen
             # Redimensionar usando PIL
            img_resized = np.array(Image.fromarray(img_array).resize(new_size, Image.LANCZOS))
            np.save(os.path.join(d_reducida, filename), img_resized)

# Se llama a la función para redimensionar imágenes
resize_npy_images(d_original, d_reducida, new_size=(480, 480))  # Llama la función para redimensionar imágenes a 480x480

# Cargar las imágenes
file_list = sorted([f for f in os.listdir(d_reducida) if f.endswith('.npy')])  # Lista y ordena los archivos .npy
print(f"Procesando {len(file_list)} imágenes")  # Imprime la cantidad de imágenes procesadas

images = [np.load(os.path.join(d_reducida, f)) for f in file_list]  # Carga los arrays de las imágenes seleccionadas

# Normalización simple
images = [(img - np.min(img)) / (np.max(img) - np.min(img) + 1e-8) for img in images]
images = [np.clip(img, 0, 1) for img in images]  # Limita los valores al rango [0, 1]

X = np.array(images[:-1])[..., np.newaxis]  # img_t  # Crea array de entrada con dimensión de canal
y = np.array(images[1:])[..., np.newaxis]   # img_t+1  # Crea array de salida (siguiente imagen) con canal

print(f"Forma de X: {X.shape}")  # Imprime la forma del array de entrada
print(f"Forma de y: {y.shape}")  # Imprime la forma del array de salida


Procesando 741 imágenes
Forma de X: (740, 480, 480, 1)
Forma de y: (740, 480, 480, 1)


Creación del modelo

In [4]:
semilla = 42
tf.random.set_seed(semilla)

model1 = Sequential([
    Conv2D(16, (3, 3), activation='relu', padding='same', input_shape=X.shape[1:]),
    Conv2D(32, (3, 3), activation='relu', padding='same'),
    Conv2D(64, (3, 3), activation='relu', padding='same'),
    Conv2D(32, (3, 3), activation='relu', padding='same'),
    Conv2D(16, (3, 3), activation='relu', padding='same'),
    Conv2D(1, (3, 3), activation='linear', padding='same')  # SIN BatchNorm en la salida
])

model1.compile(
    optimizer="adam",
    loss="mse",
    metrics=[RootMeanSquaredError(), MeanAbsolutePercentageError()]
    )

print("Resumen del modelo:")
model1.summary()

# Entrenamiento 15 épocas, batch size 8, early stopping
print("Iniciando entrenamiento")

# Guardar el mejor modelo según la métrica seleccionada
checkpoint = ModelCheckpoint(
    filepath="mejor_modelo.h5",       # Nombre del archivo de salida
    monitor="loss",                   # Métrica a monitorear ("loss" o "val_loss")
    save_best_only=True,              # Guarda solo el mejor modelo
    save_weights_only=False,          # Guarda modelo completo (estructura + pesos + optimizer)
    mode="min",                       # "min" porque queremos minimizar la loss
    verbose=1
)

early_stop = EarlyStopping(monitor='loss', patience=10, restore_best_weights=True)

history1 = model1.fit(
    X, y,
    epochs=15,
    batch_size=8,
    callbacks=[early_stop, checkpoint],
    verbose=1
)
print("¡Prueba completada!")
print(f"Pérdida final: {history1.history['loss'][-1]:.4f}")

Resumen del modelo:


/home/smoke/datac/lib/python3.12/site-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
2025-10-06 10:26:33.689445: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 480, 480, 16)   │           160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 480, 480, 32)   │         4,640 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 480, 480, 64)   │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, 480, 480, 32)   │        18,464 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_4 (Conv2D)               │ (None, 480, 480, 16)   │         4,624 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_5 (Conv2D)               │ (None, 480, 480, 1)    │           145 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 46,529 (181.75 KB)

...

93/93 ━━━━━━━━━━━━━━━━━━━━ 188s 2s/step - loss: 0.0075 - mean_absolute_percentage_error: 1255.5099 - root_mean_squared_error: 0.0868
Epoch 12/15
93/93 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - loss: 0.0076 - mean_absolute_percentage_error: 1174.6654 - root_mean_squared_error: 0.0870
Epoch 12: loss improved from 0.00753 to 0.00751, saving model to mejor_modelo.h5


93/93 ━━━━━━━━━━━━━━━━━━━━ 183s 2s/step - loss: 0.0075 - mean_absolute_percentage_error: 1252.0973 - root_mean_squared_error: 0.0867
Epoch 13/15
93/93 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - loss: 0.0076 - mean_absolute_percentage_error: 1170.9381 - root_mean_squared_error: 0.0870
Epoch 13: loss improved from 0.00751 to 0.00750, saving model to mejor_modelo.h5


93/93 ━━━━━━━━━━━━━━━━━━━━ 181s 2s/step - loss: 0.0075 - mean_absolute_percentage_error: 1248.2587 - root_mean_squared_error: 0.0866
Epoch 14/15
93/93 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - loss: 0.0076 - mean_absolute_percentage_error: 1168.4440 - root_mean_squared_error: 0.0869
Epoch 14: loss improved from 0.00750 to 0.00750, saving model to mejor_modelo.h5


93/93 ━━━━━━━━━━━━━━━━━━━━ 182s 2s/step - loss: 0.0075 - mean_absolute_percentage_error: 1245.4829 - root_mean_squared_error: 0.0866
Epoch 15/15
93/93 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - loss: 0.0075 - mean_absolute_percentage_error: 1165.5149 - root_mean_squared_error: 0.0868
Epoch 15: loss improved from 0.00750 to 0.00748, saving model to mejor_modelo.h5


93/93 ━━━━━━━━━━━━━━━━━━━━ 181s 2s/step - loss: 0.0075 - mean_absolute_percentage_error: 1243.2931 - root_mean_squared_error: 0.0865
¡Prueba completada!
Pérdida final: 0.0075


Valores de las metricas

In [7]:
from tensorflow.keras.models import load_model

# Cargar el mejor modelo guardado sin compilar (para evitar problemas de deserialización de métricas)
mejor_modelo = load_model("mejor_modelo.h5", compile=False)

# Recompilar el modelo con las métricas correctas
mejor_modelo.compile(
	optimizer="adam",
	loss="mse",
	metrics=[tf.keras.metrics.RootMeanSquaredError(), tf.keras.metrics.MeanAbsolutePercentageError()]
)

print("Mejor modelo cargado correctamente")
mejor_modelo.summary()

# Evaluación en los mismos datos de entrenamiento (o validación si tienes)
loss, rmse, mape = mejor_modelo.evaluate(X, y, verbose=1)
print(f"Resultados del mejor modelo:")
print(f" - Loss (MSE): {loss:.4f}")
print(f" - RMSE: {rmse:.4f}")
print(f" - MAPE: {mape:.2f}%")

# Predicciones con el mejor modelo
y_pred_best = mejor_modelo.predict(X)

# Ejemplo: mostrar primeras 5 predicciones
print("Ejemplo de predicciones (primeros 5):")
print(y_pred_best[:5])

Mejor modelo cargado correctamente


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 480, 480, 16)   │           160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 480, 480, 32)   │         4,640 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 480, 480, 64)   │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, 480, 480, 32)   │        18,464 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_4 (Conv2D)               │ (None, 480, 480, 16)   │         4,624 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_5 (Conv2D)               │ (None, 480, 480, 1)    │           145 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 46,529 (181.75 KB)

 Trainable params: 46,529 (181.75 KB)

 Non-trainable params: 0 (0.00 B)

24/24 ━━━━━━━━━━━━━━━━━━━━ 19s 767ms/step - loss: 0.0077 - mean_absolute_percentage_error: 1283.4808 - root_mean_squared_error: 0.0878
Resultados del mejor modelo:
 - Loss (MSE): 0.0077
 - RMSE: 0.0878
 - MAPE: 1283.48%
24/24 ━━━━━━━━━━━━━━━━━━━━ 20s 833ms/step
Ejemplo de predicciones (primeros 5):
[[[[0.7937942 ]
   [0.95468706]
   [1.0003729 ]
   ...
   [0.9575851 ]
   [0.89119434]
   [0.96316963]]

  [[1.0300809 ]
   [0.934988  ]
   [0.92248046]
   ...
   [0.9563988 ]
   [0.9108471 ]
   [0.85288763]]

  [[1.0026902 ]
   [0.9511905 ]
   [0.95951694]
   ...
   [0.8894376 ]
   [0.876629  ]
   [0.8799485 ]]

  ...

  [[0.7870724 ]
   [0.8419538 ]
   [0.81865394]
   ...
   [0.44754142]
   [0.4619057 ]
   [0.5166113 ]]

  [[0.77872825]
   [0.87458736]
   [0.83921504]
   ...
   [0.46900782]
   [0.5257456 ]
   [0.5097383 ]]

  [[0.6581616 ]
   [0.80631024]
   [0.806968  ]
   ...
   [0.49527037]
   [0.535574  ]
   [0.5101909 ]]]


 [[[0.80111843]
   [0.9642754 ]
   [1.009563  ]
   ...
   [0.

In [ ]:
# Obtener las métricas del modelo en el conjunto de entrenamiento
loss, rmse, mape = model1.evaluate(X, y, verbose=1)
print(f"Pérdida (MSE): {loss:.4f}")
print(f"RMSE: {rmse:.4f}")
print(f"MAPE: {mape:.4f}")

Visualización de imagenes

In [ ]:
y_pred1 = model1.predict(X)  # Genera las predicciones
mostrar_resultados(
    X, y, y_pred1, n=5,
    titulo_general="Entrada vs Real vs Predicción Modelo 1 - Entrenamiento",
    xlabel="Ancho (px)", ylabel="Alto (px)"
)

In [ ]:
y_pred1 = model1.predict(X)  # Genera las predicciones

# Selecciona el número de ejemplos a mostrar
n = 5

# Muestra imágenes reales vs predichas usando un mapa de color
fig, axs = plt.subplots(n, 3, figsize=(12, 3 * n))

for i in range(n):
    axs[i, 0].imshow(X[i, ..., 0], cmap='viridis')
    axs[i, 0].set_title(f'Entrada t={i}')
    axs[i, 0].axis('off')

    axs[i, 1].imshow(y[i, ..., 0], cmap='viridis')
    axs[i, 1].set_title(f'Real t+1={i+1}')
    axs[i, 1].axis('off')

    axs[i, 2].imshow(y_pred1[i, ..., 0], cmap='viridis')
    axs[i, 2].set_title(f'Predicción t+1={i+1}')
    axs[i, 2].axis('off')

plt.tight_layout()
plt.show()

Validación

In [ ]:
# Directorio original y reducido
d_feb_1s = '/media/smoke/SSD_D0/bandas/banda13/feb_01_07_1s'
d_feb_1s_red = '/media/smoke/SSD_D0/bandas/banda13/feb_01_07_1s_red'


# Se llama a la función para redimensionar imágenes
resize_npy_images(d_feb_1s, d_feb_1s_red, new_size=(480, 480))  # Llama la función para redimensionar imágenes a 480x480

# Cargar imágenes
file_list = sorted([f for f in os.listdir(d_feb_1s_red) if f.endswith('.npy')])  # Lista y ordena los archivos .npy
#file_list = file_list[:23]  # Selecciona solo los primeros 23 archivos
print(f"Procesando {len(file_list)} imágenes")  # Imprime la cantidad de imágenes procesadas

images_val = [np.load(os.path.join(d_feb_1s_red, f)) for f in file_list]  # Carga los arrays de las imágenes seleccionadas

# Normalización simple
images_val = [(img - np.min(img)) / (np.max(img) - np.min(img) + 1e-8) for img in images_val]
images_val = [np.clip(img, 0, 1) for img in images_val]  # Limita los valores al rango [0, 1]

X_val = np.array(images_val[:-1])[..., np.newaxis]  # img_t  # Crea array de entrada con dimensión de canal
y_val = np.array(images_val[1:])[..., np.newaxis]   # img_t+1  # Crea array de salida (siguiente imagen) con canal

print(f"Forma de X: {X_val.shape}")  # Imprime la forma del array de entrada
print(f"Forma de y: {y_val.shape}")  # Imprime la forma del array de salida

Metricas de validación

In [ ]:
# Obtener las métricas del modelo en el conjunto de validación
loss, rmse, mape = model1.evaluate(X_val, y_val, verbose=1)
print(f"Pérdida (MSE): {loss:.4f}")
print(f"RMSE: {rmse:.4f}")
print(f"MAPE: {mape:.4f}")

Visualización de imagenes con conjunto de validación

In [ ]:
y_pred1_val = model1.predict(X_val)  # Genera las predicciones
mostrar_resultados(
    X_val, y_val, y_pred1_val, n=5,
    titulo_general="Entrada vs Real vs Predicción Modelo 1 - Validación",
    xlabel="Ancho (px)", ylabel="Alto (px)"
)